# 05 — Pluggable Segmentation (GEE SNIC / Local SLIC) & Pure Geo-Object Classification (GEOBIA)

This notebook demonstrates the unified `Segmenter` contract and pure **`Geo-Object Classification` (GEOBIA)** workflow in `LC_Framework`:

1. **Option A — Cloud GEE SNIC (`GEESnicSegmenter`)**: Segment regional/national Earth Engine feature stacks with overflow-safe hashing and deterministic per-tile ID striding.
2. **Option B — Local Python SLIC (`LocalSlicSegmenter`)**: Segment local multi-band GeoTIFFs with `scikit-image` (no GEE login required, [ADR-0005](../docs/adr/0005-standalone-modular-subpackages.md)).
3. **Pure `Geo-Object Classification`**: `Segment → Pool Features → Assign Labels → Classify Segments → Project Probabilities`, emitting a canonical `Class Probability Stack` (`uint16` COG + `*.manifest.json`, `paradigm="geobia"`).
4. **Shared `Segment Label Raster` Contract**: Any segment raster produced here works interchangeably in pure GEOBIA **and** in Hybrid `Classification Assembly` ([Notebook 06](06_hybrid_classification_assembly.ipynb)).

In [ ]:
# 1. Colab Bootstrap — install [obia] for local SLIC + GEOBIA (add [gee] if running GEE SNIC)
# Replace <your-org> with your company GitHub organization
# !pip install -q "lc-framework[obia] @ git+https://github.com/<your-org>/LC_Framework.git"
# Optional if running Section 3A (GEE SNIC):
# !pip install -q "lc-framework[gee,obia] @ git+https://github.com/<your-org>/LC_Framework.git"

from pathlib import Path
import tempfile
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import from_origin

from lc_framework.config import ClassDefinition, ClassSchema, ProjectConfig
from lc_framework.io import read_probability_stack, read_segment_raster
from lc_framework.obia import (
    GEESnicSegmenter,
    LocalSlicSegmenter,
    SLICParams,
    SNICParams,
    run_geobia_classification,
    valid_segment_mask,
)

## 2. Mount Drive & Load `Class Schema` (Self-Contained Fallback)
Loads `configs/example_project.yaml` when available, or constructs an inline `ClassSchema` so this notebook runs cleanly standalone in Google Colab without cloning the repository.

In [ ]:
# Uncomment in Google Colab:
# from google.colab import drive
# drive.mount('/content/drive')

CONFIG_PATH = Path("../configs/example_project.yaml")
if CONFIG_PATH.exists():
    cfg = ProjectConfig.from_yaml(CONFIG_PATH)
    schema = cfg.class_schema
    TARGET_CRS = cfg.crs
    TARGET_SCALE_M = cfg.scale_m
else:
    cfg = None
    TARGET_CRS = "EPSG:32648"
    TARGET_SCALE_M = 20.0
    schema = ClassSchema(
        classes=(
            ClassDefinition(id=1, name="Forest", color="#1b5e20"),
            ClassDefinition(id=2, name="Cropland", color="#fbc02d"),
            ClassDefinition(id=3, name="Waterbody", color="#0288d1"),
        )
    )

print("Class Schema:", [(c.id, c.name) for c in schema.classes])
print(f"CRS={TARGET_CRS}, scale={TARGET_SCALE_M} m")

## 3A. Option A — Cloud Segmentation with GEE SNIC (`GEESnicSegmenter`)
Use GEE SNIC for regional or national mosaics. `SNICParams` bounds raw 64-bit SNIC cluster IDs into `[1, hash_modulus]` and adds `tile_index * id_stride` per tile so exported tiles never collide or clamp at `2^31 - 1`. Set `dry_run=True` first to inspect your tile plan before launching Earth Engine tasks.

In [ ]:
RUN_GEE_SNIC = False  # Flip to True after ee.Initialize(project=...) in Colab

snic_segmenter = GEESnicSegmenter(
    SNICParams(
        size=15,                  # 15 px at 20 m -> ~300 m seed spacing
        compactness=0.8,
        connectivity=8,
        neighborhood_size=64,
        hash_modulus=200_000_000,
        id_stride=300_000_000,
    )
)

if RUN_GEE_SNIC:
    import ee
    # ee.Authenticate()
    # ee.Initialize(project="your-gcp-project")

    # Replace with your AOI bounding box (minx, miny, maxx, maxy) in EPSG:4326
    AOI_BOUNDS = (100.0, 0.0, 101.0, 1.0)
    feature_img = ee.Image("projects/your-project/assets/feature_stack_2025")
    plans, tasks = snic_segmenter.export_tiles(
        feature_img,
        bounds=AOI_BOUNDS,
        export_prefix="projects/your-project/assets/snic_segments_2025",
        tile_size_deg=0.5,
        scale_m=TARGET_SCALE_M,
        crs=TARGET_CRS,
        bands=("red", "nir", "swir1", "NDVI"),
        destination="drive",
        drive_folder="LC_Framework_Exports",
        dry_run=True,  # Set False to submit tasks
    )
    print(f"Planned {len(plans)} SNIC tiles (submitted={len(tasks)})")
else:
    print("GEE SNIC skipped (RUN_GEE_SNIC=False). Proceeding with local SLIC + GEOBIA.")

## 3B. Option B — Local Python Segmentation (`LocalSlicSegmenter`) & Pure GEOBIA
Point `FEATURE_TIF` and `SAMPLES_PATH` to your local feature stack GeoTIFF and reference samples (either a plain `.csv` with `x, y, class_id` or a vector `.gpkg`/`.geojson`).

If you already exported a GEE SNIC `Segment Label Raster` from Section 3A, set `PRECOMPUTED_SEGMENT_TIF = Path(...)` — `run_geobia_classification(...)` will automatically run `repair_gee_int32_overflow` on read and align it to your feature stack.

In [ ]:
FEATURE_TIF = Path("data/feature_stack_2025.tif")
SAMPLES_PATH = Path("data/training_points.csv")          # CSV (x, y, class_id) or .gpkg/.geojson
PRECOMPUTED_SEGMENT_TIF = None                            # Optional: Path("data/snic_segments_2025.tif")
OUTPUT_DIR = Path("outputs/geobia_2025")

slic_segmenter = LocalSlicSegmenter(
    SLICParams(
        n_segments=500,
        compactness=5.0,
        sigma=1.0,
        start_label=1,
        normalize_bands=True,
    )
)

outputs = None
if FEATURE_TIF.exists() and SAMPLES_PATH.exists():
    outputs = run_geobia_classification(
        FEATURE_TIF,
        SAMPLES_PATH,
        OUTPUT_DIR,
        class_schema=schema,
        segment_raster_path=PRECOMPUTED_SEGMENT_TIF,
        segmenter=slic_segmenter,
        include_std=True,
        include_morphology=True,
        min_purity=0.6,
        n_estimators=200,
        epoch="2025",
    )
    print("Segment Label Raster:", outputs.segment_raster)
    print("Class Probability Stack:", outputs.probabilities)
    print(
        f"Pooled {len(outputs.segment_features)} geo-objects "
        f"({outputs.classification.n_labeled_segments} labeled for training)."
    )
else:
    print(
        "Real project files not found on disk — run Section 4 below for a self-contained "
        "synthetic GEOBIA walkthrough, or set FEATURE_TIF and SAMPLES_PATH above."
    )

## 4. Self-Contained Synthetic Walkthrough (Runs Out of the Box in Colab)
If you haven't uploaded your own rasters yet, this cell synthesizes a small 3-band feature GeoTIFF and point training table in a temporary directory and runs the full local SLIC + pure `Geo-Object Classification` pipeline end to end.

In [ ]:
if outputs is None:
    demo_dir = Path(tempfile.mkdtemp(prefix="lc_geobia_demo_"))
    rng = np.random.default_rng(42)
    h, w = 24, 24
    cube = np.zeros((3, h, w), dtype=np.float32)
    cube[:, :12, :] = np.array([[[0.05]], [[0.45]], [[0.12]]]) + rng.normal(0, 0.005, (3, 12, w))
    cube[:, 12:, :12] = np.array([[[0.18]], [[0.30]], [[0.35]]]) + rng.normal(0, 0.005, (3, 12, 12))
    cube[:, 12:, 12:] = np.array([[[0.04]], [[0.02]], [[0.01]]]) + rng.normal(0, 0.003, (3, 12, 12))

    demo_transform = from_origin(500000.0, 1000000.0, 10.0, 10.0)
    demo_feat = demo_dir / "synthetic_features.tif"
    with rasterio.open(
        demo_feat,
        "w",
        driver="GTiff",
        dtype="float32",
        count=3,
        height=h,
        width=w,
        crs="EPSG:32648",
        transform=demo_transform,
    ) as dst:
        dst.write(cube)
        for i, name in enumerate(("red", "nir", "swir1"), start=1):
            dst.set_band_description(i, name)

    demo_schema = ClassSchema(
        classes=(
            ClassDefinition(id=1, name="Forest"),
            ClassDefinition(id=2, name="Cropland"),
            ClassDefinition(id=3, name="Waterbody"),
        )
    )
    pts = [
        (3, 4, 1), (3, 12, 1), (3, 20, 1), (8, 4, 1), (8, 20, 1),
        (14, 3, 2), (14, 9, 2), (19, 3, 2), (19, 9, 2),
        (14, 15, 3), (14, 21, 3), (19, 15, 3), (19, 21, 3),
    ]
    demo_samples = pd.DataFrame(
        [
            {"x": (demo_transform * (c + 0.5, r + 0.5))[0], "y": (demo_transform * (c + 0.5, r + 0.5))[1], "class_id": cid}
            for r, c, cid in pts
        ]
    )
    outputs = run_geobia_classification(
        demo_feat,
        demo_samples,
        demo_dir / "geobia_out",
        class_schema=demo_schema,
        segmenter=LocalSlicSegmenter(SLICParams(n_segments=18, compactness=1.0, sigma=0.5)),
        n_estimators=40,
        random_state=42,
        epoch="2025",
    )
    print("Synthetic GEOBIA run complete ->", outputs.probabilities)

## 5. Inspect Outputs & Verify `Class Probability Stack` Contract
Every GEOBIA run writes a validated `uint16` COG (`[0..10000]`, nodata `65535`) paired with `geobia_probabilities.manifest.json` (`paradigm="geobia"`), ready for downstream accuracy evaluation (`lc_framework.evaluation`) or MMU/mask post-processing (`lc_framework.assembly`).

In [ ]:
if outputs is not None and outputs.probabilities.exists():
    stack = read_probability_stack(outputs.probabilities)
    seg = read_segment_raster(outputs.segment_raster)
    valid_sums = stack.probabilities[:, stack.valid_mask].sum(axis=0)
    valid_segs = seg.segments[valid_segment_mask(seg.segments)]
    print("Paradigm:", stack.manifest.paradigm)
    print("Classes:", stack.manifest.class_schema.class_names)
    print("Valid pixel probability sum range:", float(valid_sums.min()), "to", float(valid_sums.max()))
    print("Unique geo-objects in raster:", int(np.unique(valid_segs).size))